In [1]:
import pandas as pd
import numpy as np

# 행(row) 다 보기
pd.set_option('display.max_rows', None)

# 열(column) 다 보기
pd.set_option('display.max_columns', None)

### 식국외주식

In [3]:
df22 = pd.read_csv('nps/foreign_2022.csv')
df23 = pd.read_csv('nps/foreign_2023.csv')
df24 = pd.read_csv('nps/foreign_2024.csv')

In [4]:
list_종목22 = df22['종목명'].tolist()
list_종목23 = df23['종목명'].tolist()
list_종목24 = df24['종목명'].tolist()

set_종목_nm = set(list_종목22 + list_종목23 + list_종목24)

In [92]:
list_rank22, list_rank23, list_rank24 = [], [], []
list_item = []
for nm in set_종목_nm:
    rnk22, rnk23, rnk24 = None, None, None
    if nm in list_종목22:
        rnk22 = df22[df22['종목명'] == nm]['번호'].iloc[0]
    if nm in list_종목23:
        rnk23 = df23[df23['종목명'] == nm]['번호'].iloc[0]
    if nm in list_종목24:
        rnk24 = df24[df24['종목명'] == nm]['번호'].iloc[0]
    list_item.append(nm)
    list_rank22.append(rnk22)
    list_rank23.append(rnk23)
    list_rank24.append(rnk24)

df_foreign = pd.DataFrame({
    '종목명' : list_item,
    'rnk22' : list_rank22,
    'rnk23' : list_rank23,
    'rnk24' : list_rank24
})

In [93]:
mask = (df_foreign['rnk22'] > df_foreign['rnk23']) & (df_foreign['rnk23'] > df_foreign['rnk24'])
df_foreign.loc[mask, 'is_inc'] = 1

# 이유는 22년도가 기준점이고 23년은 +50% 중요하고, 24년은 +100% 중요
w22, w23, w24 = 1, 1.5, 2
df_foreign['score'] = (
    w22 / df_foreign['rnk22']
    + w23 / df_foreign['rnk23']
    + w24 /df_foreign['rnk24']
)

df_foreign['delta_22_23'] = df_foreign['rnk22'] - df_foreign['rnk23']
df_foreign['delta_23_24'] = df_foreign['rnk23'] - df_foreign['rnk24']


In [99]:
df_target = df_foreign[df_foreign.is_inc == 1].copy()
df_target.loc[:,'grp_22_23'] = df_target['delta_22_23'].apply(lambda x: '1' if x > df_target['delta_22_23'].mean() else '0')
df_target.loc[:,'grp_23_24'] = df_target['delta_23_24'].apply(lambda x: '1' if x > df_target['delta_23_24'].mean() else '0')
df_target.loc[:,'cluster'] = df_target['grp_22_23'] + df_target['grp_23_24']

df_final_target = df_target[df_target.cluster == '11'].copy()
df_final_target['grp_22_23'] = df_final_target['delta_22_23'].apply(
        lambda x: '1' if x > df_final_target['delta_22_23'].mean() else '0')
df_final_target['grp_23_24'] = df_final_target['delta_23_24'].apply(
        lambda x: '1' if x > df_final_target['delta_23_24'].mean() else '0')
df_final_target['cluster'] = df_final_target['grp_22_23'] + df_final_target['grp_23_24']
df_final_target[df_final_target.cluster == '11'].sort_values(by='score', ascending=False)


,종목명,rnk22,rnk23,rnk24,is_inc,score,delta_22_23,delta_23_24,grp_22_23,grp_23_24,cluster
849,PALANTIR TECHNOLOGIES INC A,2150.0,625.0,110.0,1.0,0.021047,1525.0,515.0,1,1,11
1141,CARNIVAL CORP,2405.0,968.0,358.0,1.0,0.007552,1437.0,610.0,1,1,11
2126,TRIP.COM GROUP LTD,2083.0,998.0,510.0,1.0,0.005905,1085.0,488.0,1,1,11
2273,MONCLER SPA,2166.0,1094.0,564.0,1.0,0.005379,1072.0,530.0,1,1,11
1728,PURE STORAGE INC CLASS A,2026.0,1163.0,613.0,1.0,0.005046,863.0,550.0,1,1,11
3639,ZILLOW GROUP INC C,2763.0,1246.0,769.0,1.0,0.004167,1517.0,477.0,1,1,11
1163,WIX.COM LTD,2835.0,1663.0,940.0,1.0,0.003382,1172.0,723.0,1,1,11
3666,ASE TECHNOLOGY HOLDING ADR,2760.0,1825.0,1002.0,1.0,0.003180,935.0,823.0,1,1,11
1074,CELESTICA INC,2957.0,2095.0,1140.0,1.0,0.002809,862.0,955.0,1,1,11
63,JAPAN POST BANK CO LTD,3080.0,2294.0,1394.0,1.0,0.002413,786.0,900.0,1,1,11


In [104]:
df_final_target[df_final_target.cluster == '01'].sort_values(by='score', ascending=False)

,종목명,rnk22,rnk23,rnk24,is_inc,score,delta_22_23,delta_23_24,grp_22_23,grp_23_24,cluster
205,NU HOLDINGS LTD/CAYMAN ISL A,2456.0,1917.0,345.0,1.0,0.006987,539.0,1572.0,0,1,01
1206,EMCOR GROUP INC,1934.0,1563.0,548.0,1.0,0.005126,371.0,1015.0,0,1,01
2734,MANHATTAN ASSOCIATES INC,2231.0,1706.0,867.0,1.0,0.003634,525.0,839.0,0,1,01
398,GODREJ PROPERTIES LTD,2210.0,1774.0,1227.0,1.0,0.002928,436.0,547.0,0,1,01
486,ASAHI KASEI CORP,2772.0,2082.0,1265.0,1.0,0.002662,690.0,817.0,0,1,01
4214,NAURA TECHNOLOGY GROUP CO A,2726.0,2204.0,1517.0,1.0,0.002366,522.0,687.0,0,1,01
389,SOUTHWEST AIRLINES CO,2829.0,2315.0,1696.0,1.0,0.002181,514.0,619.0,0,1,01
697,BRF SA,2837.0,2420.0,1665.0,1.0,0.002174,417.0,755.0,0,1,01
2034,OFFSHORE OIL ENGINEERING A,2897.0,2257.0,1746.0,1.0,0.002155,640.0,511.0,0,1,01
3018,TIS INC,2973.0,2669.0,1698.0,1.0,0.002076,304.0,971.0,0,1,01


----

- 24년도에 신규로 유입된 회사가 있는가?

In [103]:
df_foreign[df_foreign['rnk22'].isnull() & df_foreign['rnk23'].isnull()].sort_values(by='rnk24', ascending=True).head(10)

,종목명,rnk22,rnk23,rnk24,is_inc,score,delta_22_23,delta_23_24
2738,APPLOVIN CORP CLASS A,NaN,NaN,75.0,NaN,NaN,NaN,NaN
1508,GENERAL ELECTRIC,NaN,NaN,85.0,NaN,NaN,NaN,NaN
4020,NIKE INC CL B,NaN,NaN,134.0,NaN,NaN,NaN,NaN
637,T ROWE PRICE US EQUITY RESEARC,NaN,NaN,167.0,NaN,NaN,NaN,NaN
1399,GE VERNOVA INC,NaN,NaN,185.0,NaN,NaN,NaN,NaN
1373,MUENCHENER RUECKVERSICHERUNG,NaN,NaN,291.0,NaN,NaN,NaN,NaN
2547,TE CONNECTIVITY PLC,NaN,NaN,336.0,NaN,NaN,NaN,NaN
3521,FERGUSON ENTERPRISES INC,NaN,NaN,349.0,NaN,NaN,NaN,NaN
1334,T ROWE PRICE GROWTH STOCK ETF,NaN,NaN,370.0,NaN,NaN,NaN,NaN
3610,WILLIAMS SONOMA INC,NaN,NaN,402.0,NaN,NaN,NaN,NaN
